# IndianConstitution — Getting Started

A comprehensive introduction to the `indianconstitution` Python library.

This notebook covers:
1. Installation
2. Article retrieval
3. Keyword search
4. Preamble access
5. Data export
6. Graph analysis
7. pandas integration

## 1. Installation

Install the core package (zero external dependencies beyond Pydantic):

```bash
pip install indianconstitution
```

For data-science features (pandas, NetworkX):
```bash
pip install "indianconstitution[data]"
```

In [ ]:
# Uncomment to install:
# !pip install -q "indianconstitution[data]"

## 2. Core Imports & Setup

In [ ]:
from indianconstitution import get_article, get_constitution, search

ic = get_constitution()
print(f"Loaded: {ic}")
print(f"Total articles: {len(ic)}")

## 3. Article Retrieval

Retrieve any article by its number — including articles with letter suffixes like `21A`.

In [ ]:
# Retrieve by number
article = get_article("21A")
print(f"Article {article.number}: {article.title}")
print(f"\nFull text:\n{article.content}")

In [ ]:
# Another example — Right to Equality
art14 = get_article("14")
print(f"Article {art14.number}: {art14.title}")
print(f"Text: {art14.text}")  # .text is an alias for .content

## 4. Keyword Search

The library uses an **inverted-index engine** for sub-millisecond lookups.
All query tokens must appear in the article (AND logic).

In [ ]:
results = search("right to equality", limit=5)
for r in results:
    print(f"  [{r.number}] {r.title}")

In [ ]:
# Search for specific topics
for r in search("untouchability"):
    print(f"  [{r.number}] {r.title}")
    print(f"    Preview: {r.content[:150]}...\n")

## 5. The Preamble

In [ ]:
print(ic.preamble)

## 6. Cross-Reference Graph Analysis

The library builds a **directed graph** of inter-article references.
Requires `networkx` (`pip install "indianconstitution[data]"`).

In [ ]:
related = ic.get_related_articles("32")
print("Article 32 references   :", related["references"])
print("Articles referencing 32 :", related["referenced_by"])

In [ ]:
import networkx as nx

G = ic.get_graph()
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

centrality = nx.degree_centrality(G)
top_5 = sorted(centrality, key=centrality.get, reverse=True)[:5]
print("\nMost referenced articles:")
for art in top_5:
    a = ic.get_article(art)
    print(f"  Art. {art}: {a.title if a else 'N/A'}  (centrality: {centrality[art]:.4f})")

In [ ]:
# PageRank analysis
top_pr = ic.get_central_articles(limit=10)
print("Top 10 articles by PageRank:")
for art_num, score in top_pr:
    a = ic.get_article(art_num)
    print(f"  Art. {art_num}: {a.title if a else 'N/A'}  (score: {score:.6f})")

## 7. pandas Integration

In [ ]:
import pandas as pd

df = ic.to_dataframe()
print(f"DataFrame: {df.shape[0]} rows × {df.shape[1]} columns")
df[["number", "title", "part"]].head(10)

In [ ]:
# Articles per Part
df["part"].value_counts().sort_index().head(20)

## 8. Export

In [ ]:
ic.export("json", "constitution_export.json")
ic.export("csv", "constitution_export.csv")
ic.export("markdown", "constitution_export.md")
print("✓ All exports complete")